# QuickCart Intent Classification: Comprehensive Exploratory Data Analysis (EDA)

This interactive notebook walks through the end-to-end dataset audit, profiling, noise quantification, taxonomy analysis, clustering, and silver label noise estimation.

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configure plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Data Integrity & Volume Check

In [ ]:
integrity = pd.read_csv('../results/eda/integrity_checks.csv')
integrity

## 2. Intent Class Distribution & Head-to-Tail Imbalance

In [ ]:
label_counts = pd.read_csv('../results/eda/label_counts.csv')
display(label_counts)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(label_counts))
width = 0.4
ax.bar(x - width/2, label_counts['train_count'], width, label='Train', color='#1e3a8a')
ax.bar(x + width/2, label_counts['dev_count'] * (30000/4000), width, label='Dev (Scaled 7.5x)', color='#0284c7', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(label_counts['intent'], rotation=45, ha='right')
ax.set_ylabel('Utterance Count')
ax.set_title('Intent Distribution Comparison (Train vs Dev Normalized)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Message Length Profiling

In [ ]:
length_profile = pd.read_csv('../results/eda/profile_length.csv')
display(length_profile[length_profile['slice'].str.startswith('split_')])

## 4. Noise Scorecard across Splits

In [ ]:
noise_sc = pd.read_csv('../results/eda/noise_scorecard.csv')
display(noise_sc)

# Visualize noise shares
share_cols = [c for c in noise_sc.columns if c.endswith('_share')]
clean_labels = [c.replace('flag_', '').replace('_share', '').title() for c in share_cols]
fig, ax = plt.subplots(figsize=(10, 5))
noise_sc.set_index('split')[share_cols].plot(kind='bar', ax=ax, colormap='tab10')
ax.set_xticklabels(noise_sc['split'], rotation=0)
ax.set_ylabel('Fraction of Messages')
ax.set_title('Prevalence of 6 Noise Types across Splits')
ax.legend(clean_labels, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Taxonomy Health & Overlapping Intent Pairs

In [ ]:
tax_health = pd.read_csv('../results/eda/taxonomy_health.csv')
display(tax_health[['label', 'n_taxonomy_examples', 'train_support', 'top_overlapping_label', 'recommended_action']])

overlap_pairs = pd.read_csv('../results/eda/intent_overlap_pairs.csv')
print('Top 10 Closest Intent Pairs by Train Cosine Similarity:')
display(overlap_pairs.head(10))

## 6. Unsupervised Clustering Discoveries

In [ ]:
clusters = pd.read_csv('../results/eda/cluster_summary.csv')
display(clusters.head(15))

crosstab = pd.read_csv('../results/eda/cluster_vs_label_crosstab.csv', index_col=0)
fig, ax = plt.subplots(figsize=(12, 8))
cax = ax.matshow(crosstab, cmap='Blues')
fig.colorbar(cax)
ax.set_xticks(range(len(crosstab.columns)))
ax.set_xticklabels(crosstab.columns, rotation=90)
ax.set_yticks(range(len(crosstab.index)))
ax.set_yticklabels([f'Cluster {i}' for i in crosstab.index])
ax.set_title('Cluster vs Intent Label Crosstab Heatmap', pad=20)
plt.tight_layout()
plt.show()

## 7. Silver Label Noise Estimation & High-Risk Queue

In [ ]:
noise_est = pd.read_csv('../results/eda/label_noise_estimate.csv')
display(noise_est)

queue = pd.read_csv('../results/eda/label_quality_queue.csv')
print(f'Total High-Risk Rows Flagged: {len(queue)}')
display(queue[['assigned_label', 'predicted_label', 'confidence_gap', 'utterance']].head(10))